### Data Connect Hub — SDK quickstart

Runs the SDK against a live Data Connect Hub deployment.

**Prerequisites**

1. A Data Connect Hub cluster is up and running with the gateway accessible from your machine.
2. A **connector secret** with credentials for your data source already exists in the cluster.
3. Use the `sdk/python/.venv` kernel for this notebook.

**Configuration**

Update the environment variables below to match the gateway host and credentials provided by your cluster administrator:

In [ ]:
import os

DCH_HOST = os.getenv("DCH_HOST", "rh-ai.apps.dch-dev-pool-7s9kq.aws.rh-ods.com")
NAMESPACE = os.getenv("DCH_NAMESPACE", "opendatahub")
TENANT_ID = os.getenv("DCH_TENANT_ID", "")  # namespace where the connector secret and data connections reside
DB_SECRET_NAME = os.getenv("DCH_DB_SECRET_NAME", "my-db-creds")
INSECURE = os.getenv("DCH_INSECURE", "true").lower() in ("1", "true", "yes")
CA_CERT = os.getenv("DCH_CA_CERT") or None

# Only needed if using the oc CLI for token retrieval (Option 1 below)
SA_NAME = os.getenv("DCH_SA_NAME", "dch-user")
SA_ISSUER = os.getenv("DCH_SA_ISSUER", "https://kubernetes.default.svc")

In [ ]:
import subprocess

from data_connect_hub import DataConnectClient


# Option 1: use the oc CLI to request and auto-refresh tokens
def sa_token() -> str:
    return subprocess.check_output(
        ["oc", "create", "token", SA_NAME, "-n", NAMESPACE,
         "--audience", SA_ISSUER, "--duration", "3600s"],
        text=True,
    ).strip()

# Option 2: paste a static token instead
# TOKEN = os.getenv("DCH_TOKEN", "")

client = DataConnectClient(
    url=DCH_HOST,
    token_provider=sa_token,  # for Option 2, replace with: token=TOKEN,
    tenant_id=TENANT_ID,
    ca_cert=CA_CERT,
    insecure=INSECURE,
)
print(f"Client ready — gateway: {DCH_HOST}")

**Connection types (REST)**

In [ ]:
types = client.list_connection_types()
print(f"Found {len(types)} connection type(s):")
for ct in types:
    print(f"  [{ct.id}] {ct.name}: {ct.description}")

# client.delete_connection_type("6d577a9c-337c-4766-8f85-8c025550c829")

In [ ]:
new_type = client.create_connection_type(
    name="example-postgres",
    provider="postgres",
    description="PostgreSQL connector created by quickstart",
)
print(f"Created type: {new_type.id} ({new_type.name})")

In [ ]:
fetched = client.get_connection_type(new_type.id)
print(f"Fetched: {fetched.name} — {fetched.description}")

**Connections (REST)**

Connections reference a connection type. The cell above must have run first.

In [ ]:
connections = client.list_connections()
print(f"Found {len(connections)} connection(s):")
for c in connections:
    print(f"  [{c.id}] {c.name} (type={c.data_connection_type_id}, format={c.format})")

In [ ]:
from data_connect_hub import AdminSecretRef

new_conn = client.create_connection(
    name="quickstart-db",
    connection_type_id=new_type.id,
    data_format="tabular",
    admin=AdminSecretRef(secret_ref=DB_SECRET_NAME),
)
print(f"Created connection: {new_conn.id} ({new_conn.name})")

In [ ]:
fetched_conn = client.get_connection(new_conn.id)
print(f"Fetched: {fetched_conn.name} (type={fetched_conn.data_connection_type_id}, format={fetched_conn.format})")

**Flight SQL**

Query data through the Flight SQL interface. Requires a connection whose database is reachable from the Flight service.

In [ ]:
info = client.server_info()
print("Server info:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
CONNECTION_ID = new_conn.id  # or paste an existing connection ID
QUERY = "SELECT 1 AS hello"  # replace with a query for your database

table = client.read(QUERY, CONNECTION_ID)
print(f"PyArrow Table — {table.num_rows} row(s), {table.num_columns} column(s)")
print(table.to_pydict())

In [ ]:
df = client.read_pandas(QUERY, CONNECTION_ID)
df

**Cleanup**

Delete the resources created by this notebook.

In [ ]:
client.delete_connection(new_conn.id)
print(f"Deleted connection: {new_conn.id}")

In [ ]:
client.delete_connection_type(new_type.id)
print(f"Deleted connection type: {new_type.id}")